# Haul/Dump Truck: electric wheel-motor bearing, sensor vs computer vision

Saskatchewan Polytechnic PROJ 613, Group 9 (Ganesh Mishra, Lakshya Sharma, Sagar Patel). **Equipment #3.**

| | Sensor branch | Computer-vision branch |
|---|---|---|
| Data | UODS-VAFDC accelerometer (University of Ottawa bearing **test rig**) | XJTU-DV-Rotor event camera (**laboratory rotor-bearing rig**) |
| Classes | healthy, inner race, outer race | healthy, inner race, outer race |
| Test units | 20 recordings from 10 bearings, whole bearings held out | 9 recordings at 2000 rpm (trained on 1000 rpm); plus held-out camera angle |
| Script | `Models/haul_truck_wheel_motor_bearing.py` | same script, event-camera branch (OpenCV HOG) |

Matrix row B1: the target component is the wheel-motor rolling bearing of an **electric-drive** haul truck, not tyres, brakes or the engine. Ball faults are excluded because UODS recorded them with no load while the other classes used 400 N, which would let load stand in for fault type. The two branches come from **different rigs**: no paired samples, no fusion, and no prediction of a repair date.

In [1]:
DATA = [
    ("Sensor: UODS vibration (20 MAT)", "Raw_Data/uods_vafdc", "*.mat", 20),
    ("CV: XJTU-DV-Rotor event camera (2.8 GB zip)", "Raw_Data/xjtu_dv_rotor", "*.zip", 1),
]

import contextlib
import io
import json
import os
import platform
import runpy
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "Models").is_dir() and (p / "Preprocessing").is_dir())
RAW, PROCESSED, RESULTS = ROOT / "Raw_Data", ROOT / "Processed_Data", ROOT / "Results"


def package(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"


try:
    commit = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    commit = "unknown"
record = {"git commit": commit, "python": platform.python_version()}
record.update({n: package(n) for n in ("numpy", "scipy", "pandas", "scikit-learn", "opencv-python", "matplotlib")})
display(pd.Series(record, name="environment").to_frame())

rows = []
for label, path, pattern, expected in DATA:
    p = ROOT / path
    files = [f for f in p.rglob(pattern) if f.is_file()] if p.is_dir() else ([p] if p.is_file() else [])
    rows.append({"data": label, "location": path, "files": f"{len(files)}/{expected}",
                 "MB": round(sum(f.stat().st_size for f in files) / 1e6, 1), "local": len(files) >= expected})
availability = pd.DataFrame(rows).set_index("data")
display(availability)
LOCAL = availability["local"].to_dict()

if all(LOCAL.values()):
    for name in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):
        os.environ[name] = "http://127.0.0.1:9"  # offline guard: any network call now fails loudly
    os.environ["NO_PROXY"] = ""
    print("All data is local: running fully offline.")
else:
    print("Some data is missing: those steps are skipped. Download routes are in Documentation/capstone_details.md.")


def run_script(relative_path, *args, tail=3):
    """Run a project script exactly as `python <script>` would and show its last output lines."""
    path, buffer, saved = ROOT / relative_path, io.StringIO(), sys.argv
    sys.argv = [path.name, *args]
    try:
        with contextlib.redirect_stdout(buffer):
            runpy.run_path(str(path), run_name="__main__")
    finally:
        sys.argv = saved
    text = buffer.getvalue().replace(str(ROOT) + os.sep, "").replace(str(ROOT), ".")
    print(f"[{relative_path}]")
    for line in [l for l in text.splitlines() if l.strip()][-tail:]:
        print("  " + line)


def load(name):
    path = RESULTS / name
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else None


def metrics(block):
    keys = ("accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1")
    return {k: round(block[k], 3) for k in keys if isinstance(block.get(k), (int, float))}


def check(before, after, paths, name):
    def pick(doc, path):
        for key in path:
            doc = doc[key]
        return metrics(doc)
    same = before is not None and all(pick(before, p) == pick(after, p) for p in paths)
    print(f"{name}: {'matches' if same else 'DIFFERS from'} the committed result")

,environment
git commit,b9e72ba
python,3.11.9
numpy,2.4.6
scipy,1.17.1
pandas,2.3.3
scikit-learn,1.9.1
opencv-python,4.14.0.94
matplotlib,3.11.2


,location,files,MB,local
data,,,,
Sensor: UODS vibration (20 MAT),Raw_Data/uods_vafdc,20/20,69.0,True
CV: XJTU-DV-Rotor event camera (2.8 GB zip),Raw_Data/xjtu_dv_rotor,1/1,2812.1,True


All data is local: running fully offline.


## Data intake

`Preprocessing/haul_truck_intake.py` downloads the 20 selected UODS recordings (checked against the publisher's SHA-256) and the full XJTU-DV-Rotor zip (checked against Zenodo's MD5). Set `RUN_DOWNLOADS = True` to fetch anything missing; it needs internet.

In [2]:
RUN_DOWNLOADS = False
if RUN_DOWNLOADS:
    os.environ.pop("HTTP_PROXY", None); os.environ.pop("HTTPS_PROXY", None)
    run_script("Preprocessing/haul_truck_intake.py")
intake = load("haul_truck_intake.json")
files = pd.DataFrame(intake["sensor"]["files"])[["name", "class_code", "bearing", "stage", "source_size_bytes", "local_status"]]
display(files.groupby("class_code").agg(recordings=("name", "count"), bearings=("bearing", lambda b: sorted(set(b)))))
print("Rotor zip:", intake.get("visual", {}).get("local_status", "not downloaded"))

,recordings,bearings
class_code,,
H,10,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
I,5,"[1, 2, 3, 4, 5]"
O,5,"[6, 7, 8, 9, 10]"


Rotor zip: downloaded-and-verified


## Run the experiment

One script trains and scores both branches. **Sensor:** each 10 s recording becomes 100 windows of 0.1 s with 13 vibration features; five folds each hold out one inner-race and one outer-race bearing together with their healthy recordings. Each window is mean-removed and the raw mean is not used, because every inner-race recording carries a DC offset near +50 that is a recording artefact, not vibration. **Computer vision:** the first 2 s of each recording become eight event images with OpenCV HOG features, exactly as in the Pump notebook.

In [3]:
before = load("haul_truck_wheel_motor_bearing.json")
if all(LOCAL.values()):
    run_script("Models/haul_truck_wheel_motor_bearing.py", "--refresh", tail=2)
truck = load("haul_truck_wheel_motor_bearing.json")
if before is not None:
    check(before, truck, [("sensor", "held_out_bearings", "model"), ("visual", "train_1000_test_2000_rpm", "model")], "Haul Truck result")

[Models/haul_truck_wheel_motor_bearing.py]
  Saved Results\haul_truck_wheel_motor_bearing.json, Results\haul_truck_sensor_visual_comparison.md and Figures\haul_truck_confusions.png
  Sensor accuracy 0.900, macro F1 0.871; camera accuracy 0.444, macro F1 0.356
Haul Truck result: matches the committed result


## 1. Sensor branch: vibration

In [4]:
s = truck["sensor"]["held_out_bearings"]
display(pd.DataFrame({
    "Random forest, held-out bearings (20 recordings)": metrics(s["model"]),
    "Shuffled training labels (control)": metrics(s["shuffled_training_labels_control"]),
    "Most-frequent class (control)": metrics(s["most_frequent_control"]),
}).T)

,accuracy,macro_precision,macro_recall,macro_f1
"Random forest, held-out bearings (20 recordings)",0.9,0.914,0.867,0.871
Shuffled training labels (control),0.2,0.133,0.133,0.133
Most-frequent class (control),0.5,0.167,0.333,0.222


## 2. Computer-vision branch: event camera with OpenCV

In [5]:
v = truck["visual"]
display(pd.DataFrame({
    "Random forest, train 1000 rpm, test 2000 rpm (9)": metrics(v["train_1000_test_2000_rpm"]["model"]),
    "Held-out camera angle (18)": metrics(v["held_out_camera_angle"]["model"]),
    "Shuffled training labels (control)": metrics(v["train_1000_test_2000_rpm"]["shuffled_training_labels_control"]),
    "Most-frequent class (control)": metrics(v["train_1000_test_2000_rpm"]["most_frequent_control"]),
}).T)

,accuracy,macro_precision,macro_recall,macro_f1
"Random forest, train 1000 rpm, test 2000 rpm (9)",0.444,0.333,0.444,0.356
Held-out camera angle (18),0.444,0.300,0.444,0.357
Shuffled training labels (control),0.333,0.333,0.333,0.333
Most-frequent class (control),0.333,0.111,0.333,0.167


## 3. Sensor vs computer vision

In [6]:
comparison = pd.DataFrame([
    {"branch": "Sensor: vibration", "test recordings": s["model"]["test_recordings"], **metrics(s["model"])},
    {"branch": "CV: event camera (2000 rpm)", "test recordings": v["train_1000_test_2000_rpm"]["model"]["test_recordings"],
     **metrics(v["train_1000_test_2000_rpm"]["model"])},
    {"branch": "CV: held-out camera angle", "test recordings": v["held_out_camera_angle"]["model"]["test_recordings"],
     **metrics(v["held_out_camera_angle"]["model"])},
]).set_index("branch")[["test recordings", "accuracy", "macro_precision", "macro_recall", "macro_f1"]]
display(comparison)

,test recordings,accuracy,macro_precision,macro_recall,macro_f1
branch,,,,,
Sensor: vibration,20,0.900,0.914,0.867,0.871
CV: event camera (2000 rpm),9,0.444,0.333,0.444,0.356
CV: held-out camera angle,18,0.444,0.300,0.444,0.357


![Haul Truck confusion matrices](../Figures/haul_truck_confusions.png)

Full write-up and limits: [`Results/haul_truck_sensor_visual_comparison.md`](../Results/haul_truck_sensor_visual_comparison.md).

**Data citations (CC BY 4.0):** UODS-VAFDC, DOI 10.17632/y2px5tg92h.1; XJTU-DV, DOI 10.5281/zenodo.17447083.